
### JAB-Hessian sensitivity estimation & adaptive precision allocation



**Contents**
1. GPTQ core (shared quantizer, same as Student A's)
2. Calibration data capture (X and target attention output A(X))
3. Attention-aware joint loss (MSE + optional KL)
4. Hutchinson trace estimator
5. Greedy sensitivity-per-cost allocator
6. Shared utilities (calibration batches, perplexity)
7. Load GPT-2 + validate the from-scratch attention loss against the real model
8. JAB-Hessian: per-block sensitivity scores
9. Full pipeline: uniform GPTQ baseline
10. Full pipeline: JAB-Hessian adaptive allocation
11. Compare results


In [ ]:
!pip install -q torch transformers datasets

In [ ]:
import math
import random
import itertools

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd.functional import hessian as exact_hessian
import torch.autograd as autograd

from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

Using device: cuda


## 1. GPTQ core


In [ ]:
def collect_hessian_via_hook(model: torch.nn.Module, module: torch.nn.Module,
                              calibration_batches, device) -> torch.Tensor:
    """
    Registers a forward pre-hook on `module` (e.g. one block's attn.c_attn)
    to capture its input activations, runs `calibration_batches` through the
    *whole model* in no_grad mode, and returns the accumulated Hessian
    H = 2 X^T X for that layer.

    `calibration_batches` should be an iterable of input_ids tensors of shape
    (batch, seq_len), already on `device`.

    Returns: H, a (d_in, d_in) double-precision tensor.
    """
    d_in = module.weight.shape[0]  # Conv1D weight is (in_features, out_features)
    H = torch.zeros(d_in, d_in, dtype=torch.float64, device=device)
    n_samples = [0]

    def _hook(mod, inputs):
        x = inputs[0].detach()
        x = x.reshape(-1, x.shape[-1]).to(torch.float64)  # (tokens, d_in)
        H.add_(2.0 * x.T @ x)
        n_samples[0] += x.shape[0]

    handle = module.register_forward_pre_hook(_hook)
    try:
        model.eval()
        with torch.no_grad():
            for input_ids in calibration_batches:
                model(input_ids.to(device))
    finally:
        handle.remove()

    if n_samples[0] > 0:
        H /= n_samples[0]
    return H


def _quantize_to_grid(w_col: torch.Tensor, scale: torch.Tensor, bits: int) -> torch.Tensor:
    """
    Symmetric per-output-row fake quantization of a single input-column
    (shape: d_out) using a fixed per-row scale (shape: d_out) computed
    up front from the original weight statistics.
    """
    qmax = 2 ** (bits - 1) - 1
    q = torch.clamp(torch.round(w_col / scale), -qmax, qmax)
    return q * scale


@torch.no_grad()
def gptq_quantize_layer(weight_in_out: torch.Tensor, H: torch.Tensor, bits: int = 4,
                         damp_percent: float = 0.01, group_size: int = None,
                         act_order: bool = True, return_scale: bool = False) -> torch.Tensor:
    """
    Quantizes a weight matrix in the (d_in, d_out) "Conv1D" convention
    (GPT-2 style: forward is x @ weight) using the GPTQ algorithm.

    weight_in_out: (d_in, d_out) float tensor -- e.g. c_attn.weight.data
    H: (d_in, d_in) Hessian from collect_hessian_via_hook
    bits: target bit-width
    damp_percent: Hessian damping factor for numerical stability (GPTQ default ~0.01)
    group_size: if set (e.g. 128), computes a separate per-row scale for each
        contiguous group of `group_size` input columns instead of one scale
        for the whole row. Finer granularity -> lower quantization error,
        at essentially no extra cost. None = one scale per row (previous
        default behavior).
    act_order: if True, quantizes columns in order of decreasing Hessian
        diagonal (most "sensitive" columns first, while the most
        compensation budget is still available) instead of naive
        left-to-right order. Standard GPTQ accuracy improvement.

    return_scale: if True, ALSO returns the exact per-(output-channel,
        group) scale tensor that was used, expressed in the ORIGINAL
        (unpermuted) column order and in the same (d_in, d_out) orientation
        as weight_in_out -- so a caller can later fake-quantize the SAME
        weight with the SAME grid (e.g. for STE fine-tuning warm-starts)
        and get back exactly what GPTQ produced, instead of silently
        re-quantizing onto a different, coarser grid.

    Returns the fake-quantized weight, same shape, and also writes it into
    weight_in_out in place. If return_scale, returns (W_final, scale) instead.
    """
    device = weight_in_out.device
    W = weight_in_out.detach().clone().to(torch.float64).T.contiguous()  # (d_out, d_in)
    d_out, d_in = W.shape

    # Damp and (optionally) reorder the Hessian before inverting.
    H = H.clone()
    mean_diag = H.diagonal().mean()
    H += damp_percent * mean_diag * torch.eye(d_in, dtype=torch.float64, device=device)

    if act_order:
        perm = torch.argsort(torch.diag(H), descending=True)
        invperm = torch.argsort(perm)
        W = W[:, perm]
        H = H[perm][:, perm]
    else:
        perm = torch.arange(d_in, device=device)
        invperm = perm

    H_inv = torch.linalg.inv(H)

    # Per-(row, group) scale, computed once from the original weights
    # (in the possibly-permuted column order) before any quantization.
    qmax = 2 ** (bits - 1) - 1
    gs = group_size if group_size is not None else d_in
    n_groups = (d_in + gs - 1) // gs
    scale = torch.zeros(d_out, n_groups, dtype=torch.float64, device=device)
    for g in range(n_groups):
        start, end = g * gs, min((g + 1) * gs, d_in)
        scale[:, g] = (W[:, start:end].abs().amax(dim=1) / qmax).clamp(min=1e-8)

    for i in range(d_in):
        row_scale = scale[:, i // gs]
        w_col = W[:, i]
        q_col = _quantize_to_grid(w_col, row_scale, bits)
        err = (w_col - q_col) / H_inv[i, i]
        if i + 1 < d_in:
            W[:, i + 1:] -= torch.outer(err, H_inv[i, i + 1:])
        W[:, i] = q_col

    if act_order:
        W = W[:, invperm]

    W_final = W.T.contiguous().to(weight_in_out.dtype)  # back to (d_in, d_out)
    weight_in_out.copy_(W_final)

    if not return_scale:
        return W_final

    # Expand the per-(row, group) scale to a per-(row, column) scale in the
    # SAME permuted column order used above, then undo act_order's
    # permutation so it lines up with weight_in_out's original columns.
    group_idx_per_permuted_col = torch.arange(d_in, device=device) // gs   # (d_in,)
    scale_per_permuted_col = scale[:, group_idx_per_permuted_col]           # (d_out, d_in)
    scale_per_original_col = scale_per_permuted_col[:, invperm] if act_order else scale_per_permuted_col
    scale_final = scale_per_original_col.T.contiguous().to(weight_in_out.dtype)  # (d_in, d_out)
    return W_final, scale_final

## 2. Calibration data capture


In [ ]:
class AttentionOutputCapture:
    """
    Captures attention output A(X) from each block, used for MSE loss.
    """
    def __init__(self, model):
        self.outputs = {}
        self.handles = []

        for idx, block in enumerate(model.transformer.h):
            # Hook before c_proj (this is A(X) before output projection)
            handle = block.attn.c_proj.register_forward_pre_hook(self._make_hook(idx))
            self.handles.append(handle)

    def _make_hook(self, idx):
        def hook(module, inputs):
            # input[0] is A(X) - the attention output
            self.outputs[idx] = inputs[0].detach().clone()
        return hook

    def remove(self):
        for h in self.handles:
            h.remove()


class ActivationCapture:
    """
    Captures input activations (X) to each block's attention.
    """
    def __init__(self, model):
        self.activations = {}
        self.handles = []

        for idx, block in enumerate(model.transformer.h):
            # Hook before c_attn (this is X, the input to attention)
            handle = block.attn.c_attn.register_forward_pre_hook(self._make_hook(idx))
            self.handles.append(handle)

    def _make_hook(self, idx):
        def hook(module, input):
            # input[0] is X
            self.activations[idx] = input[0].detach().clone()
        return hook

    def remove(self):
        for h in self.handles:
            h.remove()


def get_calibration_data(model, calibration_batch, device):
    """
    Get all calibration data for one forward pass.
    Returns X, target_A, target_attn (None for now - MSE only).
    """
    # Create captures
    act_capture = ActivationCapture(model)
    out_capture = AttentionOutputCapture(model)

    # Run model
    model.eval()
    with torch.no_grad():
        model(calibration_batch.to(device))

    # Get data
    X = act_capture.activations
    target_A = out_capture.outputs
    target_attn = None  # MSE only for now (KL can be added later)

    # Clean up
    act_capture.remove()
    out_capture.remove()

    return X, target_A, target_attn


def get_calibration_data_for_block(model, calibration_batch, device, block_idx):
    """
    Get calibration data for a specific block.
    """
    X_dict, target_A_dict, target_attn_dict = get_calibration_data(
        model, calibration_batch, device
    )

    return (
        X_dict[block_idx],
        target_A_dict[block_idx],
        None  # This will be None #target_attn_dict[block_idx]
    )

## 3. Attention-aware joint loss
`L = ||A(X) - A_hat(X)||^2 (+ lambda * KL(attention maps), optional)` -- a from-scratch, differentiable multi-head attention computation (with causal masking) as a pure function of a flattened `[W_Q | W_K | W_V]` parameter vector, which is exactly what the Hutchinson estimator needs to differentiate through.

In [ ]:
def reshape_weights(w_flat, n_embd):
    """
    Convert flattened weights back to Q, K, V matrices.
    Inputs:
        w_flat: Flattened [W_Q, W_K, W_V]
        n_embd: Model dimension (768 for GPT-2 small)
    Outputs:
        W_Q, W_K, W_V: Each of shape (n_embd, n_embd)
    """
    # Each matrix has n_embd * n_embd parameters
    size = n_embd * n_embd

    # Split into 3 parts
    W_Q = w_flat[0:size].reshape(n_embd, n_embd)
    W_K = w_flat[size:2*size].reshape(n_embd, n_embd)
    W_V = w_flat[2*size:3*size].reshape(n_embd, n_embd)

    return W_Q, W_K, W_V

def compute_attention(W_Q, W_K, W_V, X, n_head = 12, b_Q=None, b_K=None, b_V=None):
    """
    Compute attention output and attention weights.
    Inputs:
        W_Q, W_K, W_V: Weight matrices (n_embd, n_embd)
        X: Input activations (batch, seq_len, n_embd)
    Outputs:
        A_hat: Attention output (batch, seq_len, n_embd)
        attn_weights: Attention weights (batch, seq_len, seq_len)
    """
    B, T, n_embd = X.shape #batch size, sequence length, 768
    d_head = n_embd // n_head #per-head dimension
    # Compute Q, K, V
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    if b_Q is not None:
        Q = Q + b_Q
        K = K + b_K
        V = V + b_V

    Q = Q.view(B, T, n_head, d_head).transpose(1, 2)
    K = K.view(B, T, n_head, d_head).transpose(1, 2)
    V = V.view(B, T, n_head, d_head).transpose(1, 2)
    # Scaled dot-product attention
    #d_k = Q.shape[-1]
    #scale = torch.sqrt(torch.tensor(d_k, dtype=torch.float32, device=Q.device))
    scale = math.sqrt(d_head)
    scores = Q @ K.transpose(-2, -1) / scale

    causal_mask = torch.tril(torch.ones(T, T, device=X.device, dtype=torch.bool))
    scores = scores.masked_fill(~causal_mask, float("-inf"))
    # Attention weights
    attn_weights = torch.softmax(scores, dim=-1)
    # Attention output (weighted sum of values)
    #A_hat = attn_weights @ V
    A_hat = (attn_weights @ V).transpose(1, 2).contiguous().view(B, T, n_embd)

    return A_hat, attn_weights

def mse_loss(w_flat, X, target_A, n_embd, b_Q=None, b_K=None, b_V=None):
    """
    L_mse = ||A(X) - A_hat(X)||^2
    This is the primary loss function.
    """
    # Reshape and compute attention
    W_Q, W_K, W_V = reshape_weights(w_flat, n_embd)
    A_hat, _ = compute_attention(W_Q, W_K, W_V, X, b_Q=b_Q, b_K=b_K, b_V=b_V)
    # MSE loss
    return F.mse_loss(A_hat, target_A)

def kl_loss(w_flat, X, target_attn, n_embd, b_Q=None, b_K=None, b_V=None):
    """
    L_kl = KL(attention_weights || target_attention_weights)
    """
    # Reshape and compute attention
    W_Q, W_K, W_V = reshape_weights(w_flat, n_embd)
    _, attn_weights = compute_attention(W_Q, W_K, W_V, X, b_Q=b_Q, b_K=b_K, b_V=b_V)
    # KL divergence: KL(P || Q) = sum(P * log(P / Q))
    # P = target_attn, Q = attn_weights
    log_q = torch.log(attn_weights + 1e-8)  # Add epsilon for stability

    return F.kl_div(log_q, target_attn, reduction='batchmean') #batchmean = sum(KL) / batch_size (like in Q-BERT & APTQ)

def attention_loss(w_flat, X, target_A, n_embd, target_attn=None, lambda_kl=0.1, b_Q=None, b_K=None, b_V=None):
    """
    If target_attn is None: Returns MSE only O.W. :Returns MSE + lambda_kl * KL
    """
    # Always compute MSE
    loss = mse_loss(w_flat, X, target_A, n_embd, b_Q=b_Q, b_K=b_K, b_V=b_V)
    # Add KL if target attention weights are provided
    if target_attn is not None:
        kl = kl_loss(w_flat, X, target_attn, n_embd, b_Q=b_Q, b_K=b_K, b_V=b_V)
        loss = loss + lambda_kl * kl

    return loss

## 4. Hutchinson trace estimator
`trace(H) ~= (1/n) * sum(v_i^T H v_i)` for Rademacher vectors `v_i`, using two backward passes (double-backward) to get exact Hessian-vector products without ever forming the full Hessian.

In [ ]:
def hessian_vector_product(loss_fn, params, vector, retain_graph = True):
    #first order grad
    grad = autograd.grad(loss_fn(params), params, create_graph=True, retain_graph=True)[0] #retain_graph = true to keep the graph for second grad, do not release it!
    #second order grad
    hvp = autograd.grad(grad, params, grad_outputs=vector, retain_graph=True)[0]
    return hvp

def hutchinson_trace_estimator(loss_fn, params, samples=50): #number of iterations mentioned in HAWQ-V2 article
    #trace(H) ≈ (1/n) * Σ(v_i^T * H * v_i)
    if not params.requires_grad:
        params.requires_grad_(True)

    device = params.device
    estimated_trace = 0.0

    for _ in range(samples):
        #Rademacher Vector(mentioned in HAWQ-V2)
        vec = torch.randint(0, 2, params.shape, device=device) * 2 -1
        vec = vec.float()
        hvp_result = hessian_vector_product(loss_fn, params, vec)
        estimated_trace += torch.dot(vec.flatten(), hvp_result.flatten())

    return estimated_trace/samples

## 5. Greedy sensitivity-per-cost allocator
Starts every block at the highest bit-width, then repeatedly downgrades whichever block loses the least accuracy per unit of budget freed, until the budget is met -- then spends any leftover budget on the best available upgrades.

In [ ]:
#Greedy sensitivity-per-cost bit-width allocator
import random
import itertools

BIT_WIDTHS = [2, 3, 4, 8, 16] #2, 16 is deleted for now!

def generate_synthetic_scores(block_names, seed=0):
    """
    Returns a dictionary of sensitivity scores per bit width per each block.
    """
    rng = random.Random(seed)
    scores = {}

    for name in block_names:
        fragility = rng.uniform(0.5, 3.0)
        sensitivity_by_bits = {}

        for bits in BIT_WIDTHS:
            # more bits -> less sensitivity
            sensitivity = fragility / (bits ** 1.5)
            sensitivity_by_bits[bits] = sensitivity

        scores[name] = sensitivity_by_bits

    return scores

def cost(bits, param_count=1.0):
    """
    Memory cost of storing a block at a given bit-width.
    """
    return bits * param_count

def greedy_allocate(scores, budget):
    """
    scores: dict of {block_name: {bits: sensitivity}}
    budget: max total cost allowed
      1. Give every block the HIGHEST bit-width (best accuracy, most cost).
      2. While we're over budget: find the one downgrade (one block, one
         step down in bits) that saves the most cost per unit of accuracy
         lost, and apply it. Repeat.
      3. If we still have leftover budget afterward, try upgrading blocks
         back up wherever it's affordable and helps the most.
    """
    # start at max precision
    current_bits = {name: max(BIT_WIDTHS) for name in scores}

    def total_cost():
        return sum(cost(current_bits[name]) for name in scores)

    def total_sensitivity():
        return sum(scores[name][current_bits[name]] for name in scores)

    # downgrade loop
    while total_cost() > budget:
        best_block = None
        best_new_bits = None
        best_ratio = None  # (sensitivity) / (cost)

        for name in scores:
            bits_now = current_bits[name]
            lower_choices = [b for b in BIT_WIDTHS if b < bits_now]
            if not lower_choices:
                continue  # already at the lowest possible bit-width

            next_bits = max(lower_choices)
            cost_saved = cost(bits_now) - cost(next_bits)
            sensitivity_added = scores[name][next_bits] - scores[name][bits_now]

            ratio = sensitivity_added / cost_saved

            if best_ratio is None or ratio < best_ratio:
                best_ratio = ratio
                best_block = name
                best_new_bits = next_bits

        if best_block is None:
            break  # can't downgrade anything further

        current_bits[best_block] = best_new_bits

    # spend remaining budget on the best upgrade available
    made_an_upgrade = True
    while made_an_upgrade:
        made_an_upgrade = False
        best_block = None
        best_new_bits = None
        best_ratio = None  # (sensitivity) / (extra cost)

        for name in scores:
            bits_now = current_bits[name]
            higher_choices = [b for b in BIT_WIDTHS if b > bits_now]
            if not higher_choices:
                continue

            next_bits = min(higher_choices)
            extra_cost = cost(next_bits) - cost(bits_now)

            if total_cost() + extra_cost > budget:
                continue  # can't afford it

            sensitivity_saved = scores[name][bits_now] - scores[name][next_bits]
            ratio = sensitivity_saved / extra_cost

            if best_ratio is None or ratio > best_ratio:
                best_ratio = ratio
                best_block = name
                best_new_bits = next_bits

        if best_block is not None:
            current_bits[best_block] = best_new_bits
            made_an_upgrade = True

    return current_bits, total_cost(), total_sensitivity()

def brute_force_optimal(scores, budget):
    """
    Tries every possible combination of bit-widths and keeps the best one that fits the budget. Only usable for a small number of blocks.
    """
    names = list(scores.keys())
    choices_per_block = [BIT_WIDTHS] * len(names)

    best_assignment = None
    best_cost = None
    best_sensitivity = float("inf")

    for combo in itertools.product(*choices_per_block):
        total_c = sum(cost(bits) for bits in combo)
        if total_c > budget:
            continue

        total_s = sum(scores[names[i]][combo[i]] for i in range(len(names)))

        if total_s < best_sensitivity:
            best_sensitivity = total_s
            best_cost = total_c
            best_assignment = dict(zip(names, combo))

    return best_assignment, best_cost, best_sensitivity

## 6. Shared utilities
Calibration-batch construction and sliding-window perplexity evaluation (same as Student A's).

In [ ]:
import math
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

def build_calibration_batches(tokenizer, n_samples=128, seq_len=512):
    """
    Pulls `n_samples` chunks of `seq_len` tokens each from WikiText-2 train,
    as a list of (1, seq_len) input_id tensors.
    """
    raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
    text = "\n\n".join(t for t in raw["text"] if t.strip())
    ids = tokenizer(text, return_tensors="pt").input_ids[0]

    batches = []
    stride = seq_len
    for i in range(n_samples):
        start = i * stride
        if start + seq_len > ids.shape[0]:
            break
        chunk = ids[start:start + seq_len].unsqueeze(0)
        batches.append(chunk)
    return batches


@torch.no_grad()
def evaluate_perplexity(model, tokenizer, max_length=1024, stride=512):
    """
    Sliding-window perplexity on WikiText-2 test.
    """
    raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
    text = "\n\n".join(t for t in raw["text"] if t.strip())
    ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)
    seq_len = ids.shape[1]

    model.eval()
    nll_sum = 0.0
    n_tokens = 0
    prev_end = 0

    for begin in range(0, seq_len, stride):
        end = min(begin + max_length, seq_len)
        trg_len = end - prev_end
        input_ids = ids[:, begin:end]
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100

        out = model(input_ids, labels=target_ids)
        nll_sum += out.loss.item() * trg_len
        n_tokens += trg_len

        prev_end = end
        if end == seq_len:
            break

    return math.exp(nll_sum / n_tokens)

## 7. Load GPT-2 and validate the from-scratch attention loss
Before trusting JAB-Hessian scores computed from `attention_loss.py`'s reimplemented attention, confirm it reproduces GPT-2's *real* attention output almost exactly.

In [ ]:
model = AutoModelForCausalLM.from_pretrained("gpt2").to(DEVICE)
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model.eval()
n_embd = model.config.n_embd
print(f"Model dimension: {n_embd}")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Model dimension: 768


## 8. JAB-Hessian: per-block sensitivity scores
Wires the Hutchinson estimator to the real attention-output loss (Section 3) to get one sensitivity score per block, converts it to a per-bit-width sensitivity table, and runs the greedy allocator (Section 5) against it.

In [ ]:
def compute_jab_trace_from_data(model, block_idx, X, target_A, device, n_embd, samples=30):
    """
    Compute sensitivity score per attention block.
    inputs:
        model: GPT-2 model
        block_idx: Which layer
        batch: One batch of input tokens
        n_embd: Model dimension
        samples: Number of Hutchinson samples
    """
    # 1. Get weights of this block's Q, K, V
    block = model.transformer.h[block_idx]
    W = block.attn.c_attn.weight.data  # Shape: (768, 2304) = [W_Q | W_K | W_V]
    # Split into Q, K, V and flatten them into one vector
    W_Q, W_K, W_V = W.split(n_embd, dim=1)
    w_flat = torch.cat([W_Q.flatten(), W_K.flatten(), W_V.flatten()])
    w_flat.requires_grad_(True)  # for Hessian computation
    # 2. Get calibration data (X = input, target_A = attention output)
    bias = block.attn.c_attn.bias.data
    b_Q, b_K, b_V = bias.split(n_embd)
    # 3. Loss function
    def loss_fn(params):
        return attention_loss(params, X, target_A, n_embd, b_Q=b_Q, b_K=b_K, b_V=b_V)
    # 4. Compute Hessian trace using Hutchinson estimator
    trace = hutchinson_trace_estimator(loss_fn, w_flat, samples=samples)
    return trace.item()

def compute_all_jab_scores(model, batches, device, n_embd, samples=30, n_batches_to_use=4):
    """
    Compute sensitivity scores for ALL attention blocks and returns it in format of a dictionary.
    """
    n_blocks = len(model.transformer.h)
    use_batches = batches[:n_batches_to_use]
    all_traces = {f"block_{i}_QKV": [] for i in range(n_blocks)}

    print(f"\nComputing JAB scores for {n_blocks} blocks "
          f"(averaged over {len(use_batches)} batches, {samples} Hutchinson samples each)...")

    for b in use_batches:
        # ONE forward pass captures X/target_A for every block at once
        X_dict, target_A_dict, _ = get_calibration_data(model, b, device)
        for idx in range(n_blocks):
            trace = compute_jab_trace_from_data(
                model, idx, X_dict[idx], target_A_dict[idx], device, n_embd, samples
            )
            all_traces[f"block_{idx}_QKV"].append(trace)

    scores = {}
    for block_name, traces in all_traces.items():
        trace = sum(traces) / len(traces)
        scores[block_name] = trace
        level = "HIGH" if trace > 100 else "MEDIUM" if trace > 10 else "LOW"
        print(f"  {block_name}: {trace:.4f}  [{level} sensitivity]  "
              f"(min={min(traces):.4f}, max={max(traces):.4f})")

    print("Done!\n")
    return scores

def scores_to_allocator_format(jab_scores):
    """
    Convert JAB scores to format expected by greedy allocator.
    """
    allocator_input = {}
    bit_widths = [2, 3, 4, 8, 16]#[2,

    for block_name, trace in jab_scores.items():
        sensitivity_by_bits = {}
        for bits in bit_widths:
            sensitivity_by_bits[bits] = trace / (bits ** 1.5)
        allocator_input[block_name] = sensitivity_by_bits

    return allocator_input

def measure_weight_perturbation(original_W, H, bits):
    """
    HAWQ-V2 style: ||Q(W) - W||_F^2, measured directly in weight space.
    """
    w_copy = original_W.clone()
    gptq_quantize_layer(w_copy, H, bits=bits, group_size=128, act_order=True)
    perturbation = torch.norm(w_copy - original_W, p='fro') ** 2
    return perturbation.item()


def scores_to_allocator_format_hawqv2(jab_scores, model, calibration_batches, device, bit_widths=None):
    """
    HAWQ-V2 style: Omega_i(bits) = trace_i * ||Q(W_i) - W_i||_F^2,
    instead of the guessed trace / bits^1.5.
    """
    if bit_widths is None:
        bit_widths = [2, 3, 4, 8, 16]

    allocator_input = {}
    for block_name, trace in jab_scores.items():
        block_idx = int(block_name.split("_")[1])          # extract the index from "block_0_QKV" -> 0
        block = model.transformer.h[block_idx]
        c_attn = block.attn.c_attn
        original_W = c_attn.weight.data

        H = collect_hessian_via_hook(model, c_attn, calibration_batches, device)

        sensitivity_by_bits = {}
        for bits in bit_widths:
            perturbation = measure_weight_perturbation(original_W, H, bits)
            sensitivity_by_bits[bits] = trace * perturbation
        allocator_input[block_name] = sensitivity_by_bits

        print(f"  {block_name}: " + ", ".join(f"{b}bit={v:.4e}" for b, v in sensitivity_by_bits.items()))

    return allocator_input


def run_jab_allocation(model, batches, device, n_embd, target_avg_bits=4.0, samples=30, n_batches_to_use=16):

    """
    1. Compute JAB scores for all blocks
    2. Convert to allocator format (HAWQ-V2 style, measured perturbation)
    3. Run greedy allocation
    4. Return bit assignment in format of a dictionary
    """
    print("JAB-Hessian Adaptive Allocation")
    # Compute JAB scores
    print("\nComputing JAB scores...")
    jab_scores = compute_all_jab_scores(model, batches, device, n_embd, samples, n_batches_to_use)
    # Convert format
    print("Preparing for allocator...")
    allocator_input = scores_to_allocator_format_hawqv2(jab_scores, model, batches, device)
    # Set budget
    n_blocks = len(allocator_input)
    budget = target_avg_bits * n_blocks
    print(f"Budget: {budget:.1f} bits ({target_avg_bits} avg for {n_blocks} blocks)")
    # Run allocation
    print("\nRunning greedy allocation...")
    assignment, cost_used, sensitivity = greedy_allocate(allocator_input, budget)
    # results
    print("Results")
    print(f"Total cost: {cost_used:.1f} bits")
    print(f"Average bits: {cost_used / n_blocks:.2f}")
    print(f"Total sensitivity: {sensitivity:.6f}")
    print("\nPer-block allocation:")
    for block_name, bits in assignment.items():
        print(f"    {block_name}: {bits} bits")

    return assignment

## 9. Full pipeline: uniform GPTQ baseline
Quantizes every block's `c_attn` to a flat 4 bits -- the R7 comparison point for adaptive allocation. Both this and Section 10 reuse the *same* calibration batches, so the comparison is apples-to-apples.

**Note (adopted from Untitled10.ipynb's `gptq_core.py`):** like that notebook's `quantize_all_attention_blocks`, each block's Hessian is collected and quantized in sequence on the *same* live model object, so later blocks' calibration activations already reflect earlier blocks' quantization error -- this is not fully independent per-block quantization. The next cell also reuses Untitled10's saved GPTQ checkpoint (`week1_gptq_qkv_init.pt`) when present, instead of always re-running GPTQ from scratch.

In [ ]:
import os

print("Building the shared calibration set (used for all experiments below)...")
calibration_batches = build_calibration_batches(tokenizer, n_samples=128, seq_len=512)#32,128
print(f"{len(calibration_batches)} calibration batches ready.\n")

print("Loading a fresh full-precision GPT-2 for the uniform baseline...")
model_uniform = AutoModelForCausalLM.from_pretrained("gpt2").to(DEVICE)
model_uniform.eval()

# --- Checkpoint-first: reuse Untitled10.ipynb's saved GPTQ init if present on disk ---
# Untitled10's `save_quantized_qkv` step writes exactly this filename. Loading it here
# instead of re-running GPTQ avoids a redundant multi-minute pass when both notebooks
# are run in the same environment. NOTE: Untitled10's own `gptq_quantize_layer` does
# NOT support `group_size`/`act_order`, so a checkpoint it produced was quantized
# WITHOUT those refinements -- slightly different from what this cell's own GPTQ call
# below would produce. If you want this baseline to reflect this notebook's improved
# quantizer (group_size=128, act_order=True), delete the checkpoint file and let this
# cell regenerate it.
CHECKPOINT_PATH = "week1_gptq_qkv_init.pt"

if os.path.exists(CHECKPOINT_PATH):
    print(f"Found {CHECKPOINT_PATH} -- loading GPTQ-quantized Q/K/V instead of re-running GPTQ...")
    state = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    for idx, block in enumerate(model_uniform.transformer.h):
        saved = state[f"block_{idx}"]
        W_cat = torch.cat([saved["W_Q"], saved["W_K"], saved["W_V"]], dim=1).to(DEVICE)
        block.attn.c_attn.weight.data.copy_(W_cat)
    print("Loaded.\n")
else:
    print(f"No {CHECKPOINT_PATH} found -- quantizing ALL blocks to 4 bits uniformly...")
    for idx in range(len(model_uniform.transformer.h)):
        block = model_uniform.transformer.h[idx]
        c_attn = block.attn.c_attn
        print(f"  Block {idx}: quantizing to 4 bits...")
        H = collect_hessian_via_hook(model_uniform, c_attn, calibration_batches, DEVICE)
        gptq_quantize_layer(c_attn.weight.data, H, bits=4, group_size=128, act_order=True)

    # Save so this pass doesn't need to be repeated -- same format as Untitled10's Step 4,
    # so either notebook can load either notebook's checkpoint going forward.
    save_state = {}
    for idx, block in enumerate(model_uniform.transformer.h):
        W_Q, W_K, W_V = block.attn.c_attn.weight.data.split(n_embd, dim=1)
        save_state[f"block_{idx}"] = {
            "W_Q": W_Q.clone().cpu(), "W_K": W_K.clone().cpu(), "W_V": W_V.clone().cpu()
        }
    torch.save(save_state, CHECKPOINT_PATH)
    print(f"Saved GPTQ-initialized Q/K/V to {CHECKPOINT_PATH} for reuse.\n")

print("\nEvaluating perplexity...")
ppl_uniform = evaluate_perplexity(model_uniform, tokenizer)
print(f"\nUniform 4-bit perplexity: {ppl_uniform:.3f}")


Building the shared calibration set (used for all experiments below)...


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

...

128 calibration batches ready.

Loading a fresh full-precision GPT-2 for the uniform baseline...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

No week1_gptq_qkv_init.pt found -- quantizing ALL blocks to 4 bits uniformly...
  Block 0: quantizing to 4 bits...
  Block 1: quantizing to 4 bits...
  Block 2: quantizing to 4 bits...
  Block 3: quantizing to 4 bits...
  Block 4: quantizing to 4 bits...
  Block 5: quantizing to 4 bits...
  Block 6: quantizing to 4 bits...
  Block 7: quantizing to 4 bits...
  Block 8: quantizing to 4 bits...
  Block 9: quantizing to 4 bits...
  Block 10: quantizing to 4 bits...
  Block 11: quantizing to 4 bits...
Saved GPTQ-initialized Q/K/V to week1_gptq_qkv_init.pt for reuse.


Evaluating perplexity...


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.



Uniform 4-bit perplexity: 26.225


## 10. Full pipeline: JAB-Hessian adaptive allocation
Computes JAB scores, allocates bits under an average-bit budget close to the uniform baseline, applies it, and evaluates perplexity -- using the *same* calibration batches as Section 9.

**Budget note:** we use `target_avg_bits=4.3`, not `4.0`. At exactly 4.0 the budget lands on a `BIT_WIDTHS` grid point and every block saturates to the same 4-bit floor before any differentiation happens (see the sweep cell right after this one for a full explanation and a demonstration that the allocator differentiates correctly away from that point).

In [ ]:
def apply_allocation(model, assignment, calibration_batches, device):
    print("\nApplying allocation")
    for idx in range(len(model.transformer.h)):
        block_name = f"block_{idx}_QKV"
        if block_name in assignment:
            bits = assignment[block_name]
            block = model.transformer.h[idx]
            c_attn = block.attn.c_attn
            H = collect_hessian_via_hook(model, c_attn, calibration_batches, device)
            gptq_quantize_layer(c_attn.weight.data, H, bits=bits, group_size=128, act_order=True)
            print(f"  Block {idx}: {bits} bits")
    print("Allocation done.")
    return model

In [ ]:
print("Loading a fresh full-precision GPT-2 for adaptive allocation...")
model_adaptive = AutoModelForCausalLM.from_pretrained("gpt2").to(DEVICE)
model_adaptive.eval()

print("\nComputing JAB-Hessian scores and allocating bits...")
torch.manual_seed(42)
# target_avg_bits=4.3, not 4.0 -- see markdown above (4.0 sits exactly on a BIT_WIDTHS grid point)
assignment = run_jab_allocation(model=model_adaptive, batches=calibration_batches,
                                 device=DEVICE, n_embd=n_embd, target_avg_bits=4.3, samples=20)

print("\nApplying the allocation...")
model_adaptive = apply_allocation(model_adaptive, assignment, calibration_batches, DEVICE)

print("\nEvaluating perplexity...")
ppl_adaptive = evaluate_perplexity(model_adaptive, tokenizer)
print(f"\nAdaptive (JAB-Hessian) perplexity: {ppl_adaptive:.3f}")

Loading a fresh full-precision GPT-2 for adaptive allocation...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Computing JAB-Hessian scores and allocating bits...
JAB-Hessian Adaptive Allocation

Computing JAB scores...

Computing JAB scores for 12 blocks (averaged over 16 batches, 20 Hutchinson samples each)...
  block_0_QKV: 8.8141  [LOW sensitivity]  (min=8.5435, max=9.1991)
  block_1_QKV: 27.3088  [MEDIUM sensitivity]  (min=26.5417, max=28.2310)
  block_2_QKV: 97.5410  [MEDIUM sensitivity]  (min=94.5012, max=101.7118)
  block_3_QKV: 102.0859  [HIGH sensitivity]  (min=96.5775, max=107.4939)
  block_4_QKV: 68.3341  [MEDIUM sensitivity]  (min=66.5661, max=70.5521)
  block_5_QKV: 61.7517  [MEDIUM sensitivity]  (min=57.0077, max=68.5626)
  block_6_QKV: 51.4415  [MEDIUM sensitivity]  (min=46.9889, max=57.5877)
  block_7_QKV: 43.6951  [MEDIUM sensitivity]  (min=39.8768, max=47.2856)
  block_8_QKV: 43.9092  [MEDIUM sensitivity]  (min=40.1584, max=49.8791)
  block_9_QKV: 37.4549  [MEDIUM sensitivity]  (min=34.0669, max=41.5738)
  block_10_QKV: 40.3476  [MEDIUM sensitivity]  (min=36.1168, max=46.818

In [ ]:
# --- Budget sensitivity check: does the allocator actually differentiate blocks? ---
# At target_avg_bits=4.0, the budget (48 = 12 blocks x 4) lands exactly on a
# BIT_WIDTHS grid point. Below 4 bits the HAWQ-V2 perturbation term
# (||Q(W,bits)-W||_F^2) blows up steeply, so every block's "cheap" 16->8 and
# 8->4 downgrades get exhausted first regardless of its trace, and the loop
# stops the instant all 12 blocks hit the 4-bit floor -- before any block ever
# gets to compete for a downgrade below (or an upgrade above) that floor. That
# is why Section 10 above shows all 12 blocks at exactly 4 bits: it is not a
# broken allocator, it is a budget sitting exactly on that cliff.
#
# Sweeping a couple of nearby, non-grid-aligned budgets on the SAME scores
# confirms this: away from the cliff, the allocator cleanly separates
# high-trace (sensitive) blocks from low-trace ones.
print("Budget sensitivity check: reusing one JAB-score pass across a small sweep...")

model_sweep_probe = AutoModelForCausalLM.from_pretrained("gpt2").to(DEVICE)
model_sweep_probe.eval()

jab_scores_sweep = compute_all_jab_scores(model_sweep_probe, calibration_batches, DEVICE,
                                           n_embd, samples=20, n_batches_to_use=4)
allocator_input_sweep = scores_to_allocator_format_hawqv2(jab_scores_sweep, model_sweep_probe,
                                                           calibration_batches, DEVICE)

for avg_bits in [3.5, 4.3, 4.5, 6.0]:
    budget = avg_bits * len(allocator_input_sweep)
    alloc, cost_used, _ = greedy_allocate(allocator_input_sweep, budget)
    bits_used = sorted(set(alloc.values()))
    print(f"\navg_bits={avg_bits} (budget={budget:.1f}, cost_used={cost_used:.1f}):")
    print(f"  distinct bit-widths chosen: {bits_used}")
    for name, b in alloc.items():
        print(f"    {name}: {b} bits")

del model_sweep_probe

# --- Perplexity at each budget in the sweep ---
# Reuses allocator_input_sweep computed above; only the allocation + GPTQ + eval
# are redone per budget (each needs its own fresh, unquantized model).
ppl_by_budget = {}

for avg_bits in [3.5, 4.3, 4.5, 6.0]:
    budget = avg_bits * len(allocator_input_sweep)
    alloc, cost_used, _ = greedy_allocate(allocator_input_sweep, budget)

    print(f"\n=== avg_bits={avg_bits} (budget={budget:.1f}, cost_used={cost_used:.1f}) ===")
    model_budget = AutoModelForCausalLM.from_pretrained("gpt2").to(DEVICE)
    model_budget.eval()
    model_budget = apply_allocation(model_budget, alloc, calibration_batches, DEVICE)

    ppl = evaluate_perplexity(model_budget, tokenizer)
    ppl_by_budget[avg_bits] = ppl
    print(f"Perplexity at avg_bits={avg_bits}: {ppl:.3f}")

    del model_budget

print("\n=== Perplexity vs. budget summary ===")
for avg_bits, ppl in ppl_by_budget.items():
    print(f"  avg_bits={avg_bits}: perplexity={ppl:.3f}")
print(f"  (uniform 4-bit baseline: {ppl_uniform:.3f})")


Budget sensitivity check: reusing one JAB-score pass across a small sweep...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Computing JAB scores for 12 blocks (averaged over 4 batches, 20 Hutchinson samples each)...
  block_0_QKV: 8.8517  [LOW sensitivity]  (min=8.7762, max=8.9339)
  block_1_QKV: 27.3272  [MEDIUM sensitivity]  (min=26.6279, max=27.6273)
  block_2_QKV: 98.1607  [MEDIUM sensitivity]  (min=95.7791, max=101.1830)
  block_3_QKV: 102.8559  [HIGH sensitivity]  (min=96.8450, max=108.5722)
  block_4_QKV: 68.6549  [MEDIUM sensitivity]  (min=66.7665, max=71.4011)
  block_5_QKV: 61.3231  [MEDIUM sensitivity]  (min=57.5417, max=63.7591)
  block_6_QKV: 51.8534  [MEDIUM sensitivity]  (min=47.3997, max=55.1493)
  block_7_QKV: 43.6097  [MEDIUM sensitivity]  (min=40.7172, max=46.0596)
  block_8_QKV: 44.3439  [MEDIUM sensitivity]  (min=41.0073, max=48.4443)
  block_9_QKV: 37.9888  [MEDIUM sensitivity]  (min=36.2800, max=40.9364)
  block_10_QKV: 40.8333  [MEDIUM sensitivity]  (min=39.0137, max=44.1010)
  block_11_QKV: 57.9503  [MEDIUM sensitivity]  (min=55.0556, max=60.5124)
Done!

  block_0_QKV: 2bit=5.5063e

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Applying allocation
  Block 0: 3 bits
  Block 1: 3 bits
  Block 2: 4 bits
  Block 3: 4 bits
  Block 4: 4 bits
  Block 5: 4 bits
  Block 6: 4 bits
  Block 7: 3 bits
  Block 8: 3 bits
  Block 9: 3 bits
  Block 10: 3 bits
  Block 11: 4 bits
Allocation done.
Perplexity at avg_bits=3.5: 28.658

=== avg_bits=4.3 (budget=51.6, cost_used=48.0) ===


...


Applying allocation
  Block 0: 4 bits
  Block 1: 4 bits
  Block 2: 4 bits
  Block 3: 4 bits
  Block 4: 4 bits
  Block 5: 4 bits
  Block 6: 4 bits
  Block 7: 4 bits
  Block 8: 4 bits
  Block 9: 4 bits
  Block 10: 4 bits
  Block 11: 4 bits
Allocation done.
Perplexity at avg_bits=4.3: 26.225

=== avg_bits=4.5 (budget=54.0, cost_used=52.0) ===


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Applying allocation
  Block 0: 4 bits
  Block 1: 4 bits
  Block 2: 4 bits
  Block 3: 8 bits
  Block 4: 4 bits
  Block 5: 4 bits
  Block 6: 4 bits
  Block 7: 4 bits
  Block 8: 4 bits
  Block 9: 4 bits
  Block 10: 4 bits
  Block 11: 4 bits
Allocation done.
Perplexity at avg_bits=4.5: 26.204

=== avg_bits=6.0 (budget=72.0, cost_used=72.0) ===


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Applying allocation
  Block 0: 4 bits
  Block 1: 4 bits
  Block 2: 8 bits
  Block 3: 8 bits
  Block 4: 8 bits
  Block 5: 8 bits
  Block 6: 8 bits
  Block 7: 4 bits
  Block 8: 4 bits
  Block 9: 4 bits
  Block 10: 4 bits
  Block 11: 8 bits
Allocation done.
Perplexity at avg_bits=6.0: 25.184

=== Perplexity vs. budget summary ===
  avg_bits=3.5: perplexity=28.658
  avg_bits=4.3: perplexity=26.225
  avg_bits=4.5: perplexity=26.204
  avg_bits=6.0: perplexity=25.184
  (uniform 4-bit baseline: 26.225)


## 11. Joint attention-aware fine-tuning (fills in Untitled10's `joint_calibration_skeleton`)

Sections 8-10 only used `attention_loss` as an importance **score** (the JAB-Hessian trace) to decide *how many bits* each block gets -- the actual quantized weights were still produced by GPTQ minimizing the conventional `||XW - XW_hat||^2`, not the attention-output loss the PDF's Objective 1 asks for.

This section closes that gap: it fills in the interface Untitled10.ipynb sketches in `joint_calibration_skeleton` ("Week 2 task -- once JAB-Hessian is wired in"), using this notebook's own validated `attention_loss`/`compute_attention`. Each block's GPTQ-initialized `W_Q, W_K, W_V` are wrapped in a straight-through quantizer and fine-tuned with Adam to directly minimize `L(A(X), A_hat(X))`, jointly -- i.e. `min_{W_hat_Q,W_hat_K,W_hat_V} L(A(X), A_hat(X))` from the PDF, not a proxy for it.

**Correctness note:** `target_A` is always captured from a separate, untouched full-precision `model_float` -- never from `model_joint`, which is being progressively quantized. Distilling a quantized model toward its own (already-degraded) output would silently defeat the objective.

---

### Bug found after running this section: perplexity got *worse* (26.225 -> 32.660)

**Root cause.** `STEQuantize`'s fake-quantization grid is controlled by a `scale`
tensor, and the fine-tuning loop below used to build that scale with
`per_row_scale_flat`, which computes **one scale per input dimension, shared
across all 768 output channels, with no grouping at all**. But the GPTQ warm
start it fine-tunes from (`gptq_quantize_layer(..., group_size=128,
act_order=True)`) quantizes with **one scale per output channel, per group of
128 input columns** -- a completely different axis and granularity.

So the very first call to `ste_quantize(w_flat, scale_flat, bits)` inside the
fine-tuning loop -- *before Adam ever takes a step* -- silently re-quantized
the carefully-optimized GPTQ solution onto a coarser, misaligned grid. A
synthetic test with the exact same shapes/settings confirms this introduces
~9-10% relative weight error immediately at step 0 (see the sanity-check cell
below). Only 8 tiny (`lr=1e-4`) Adam steps per block can't repair that kind of
damage, and because blocks are quantized *sequentially* (block *i*'s
calibration input `X` already reflects blocks `0..i-1`'s quantization error),
the damage compounds -- consistent with block 0's fine-tuned loss (0.0008)
being ~56x smaller than block 11's (0.043) in the original run.

**Fix**, implemented below:
1. `gptq_quantize_layer` gained an optional `return_scale=True` mode that
   hands back the *exact* per-(output-channel, group) scale it used, in the
   original column order. `joint_attention_aware_finetune` now does its own
   GPTQ warm start (instead of a separate pre-quantization pass) and reuses
   that exact scale, so `ste_quantize` reproduces the GPTQ solution exactly
   at step 0 -- fine-tuning can now only move *away* from a correct starting
   point, never a silently-corrupted one.
2. **Best-iterate tracking + a safety net**: each block keeps the
   lowest-loss iterate seen during its (short, noisy) Adam trajectory, and
   only commits it if it actually beats the GPTQ-only starting loss;
   otherwise the block keeps its GPTQ-only weights. This gives a formal
   per-block guarantee that joint fine-tuning cannot make the *local*
   attention-reconstruction objective worse than plain GPTQ.
3. Gradient clipping, and calibration batches that rotate across the
   calibration set instead of every block reusing the same first 8 batches.


In [ ]:
# --- Sanity check: does the fine-tuning scale match the GPTQ warm-start scale? ---
# Synthetic stand-in for one block's c_attn, small enough to run instantly,
# with the SAME shapes/settings pattern (group_size, act_order, bits) as the
# real pipeline. Demonstrates the bug (OLD per_row_scale_flat) and the fix
# (NEW: reuse gptq_quantize_layer's own return_scale) side by side.

def _demo_per_row_scale_flat_OLD(W0, qmax):
    """The buggy scale used by the original Section 11: one scale per INPUT
    row, shared across every output column, no grouping at all."""
    row_max = W0.detach().abs().amax(dim=1, keepdim=True).clamp(min=1e-8)
    return (row_max / qmax).expand_as(W0).reshape(-1)

torch.manual_seed(0)
demo_n_embd, demo_group_size, demo_bits = 64, 32, 4  # small stand-ins for 768 / 128 / 4

W_demo = torch.randn(demo_n_embd, 3 * demo_n_embd) * 0.05
X_demo = torch.randn(2000, demo_n_embd)
H_demo = (2.0 * X_demo.T @ X_demo / X_demo.shape[0]).to(torch.float64)

W_after, scale_correct = gptq_quantize_layer(
    W_demo.clone(), H_demo, bits=demo_bits, group_size=demo_group_size,
    act_order=True, return_scale=True,
)
W_Q0, W_K0, W_V0 = W_after.split(demo_n_embd, dim=1)
w_flat_demo = torch.cat([W_Q0.flatten(), W_K0.flatten(), W_V0.flatten()])
qmax_demo = 2 ** (demo_bits - 1) - 1

# OLD (buggy): re-quantizing the GPTQ solution with the wrong-axis scale
scale_old = torch.cat([_demo_per_row_scale_flat_OLD(w, qmax_demo) for w in (W_Q0, W_K0, W_V0)])
w_old = torch.clamp(torch.round(w_flat_demo / scale_old), -qmax_demo, qmax_demo) * scale_old
err_old = ((w_old - w_flat_demo).norm() / w_flat_demo.norm()).item()

# NEW (fixed): reusing GPTQ's own exact scale
scale_Q, scale_K, scale_V = scale_correct.split(demo_n_embd, dim=1)
scale_new = torch.cat([scale_Q.flatten(), scale_K.flatten(), scale_V.flatten()])
w_new = torch.clamp(torch.round(w_flat_demo / scale_new), -qmax_demo, qmax_demo) * scale_new
err_new = ((w_new - w_flat_demo).norm() / w_flat_demo.norm()).item()

print(f"Relative error re-quantizing the GPTQ solution with the OLD scale : {err_old:.4%}")
print(f"Relative error re-quantizing the GPTQ solution with the NEW scale : {err_new:.6%}")
if err_old > 0.01 and err_new < 1e-4:
    print("\nConfirmed: the OLD scale silently corrupts the GPTQ warm start; the NEW scale is exact.")


Relative error re-quantizing the GPTQ solution with the OLD scale : 9.5696%
Relative error re-quantizing the GPTQ solution with the NEW scale : 0.000003%

Confirmed: the OLD scale silently corrupts the GPTQ warm start; the NEW scale is exact.


In [ ]:
class STEQuantize(torch.autograd.Function):
    """
    Straight-through estimator for fake quantization: rounds onto the bit-grid
    in the forward pass (so downstream loss "sees" quantization error), but
    passes the incoming gradient through unchanged in the backward pass, so
    Adam can adjust the underlying float weights to compensate.
    """
    @staticmethod
    def forward(ctx, w, scale, bits):
        qmax = 2 ** (bits - 1) - 1
        q = torch.clamp(torch.round(w / scale), -qmax, qmax)
        return q * scale

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output, None, None


def ste_quantize(w, scale, bits):
    return STEQuantize.apply(w, scale, bits)


def joint_attention_aware_finetune(model_joint, model_float, calibration_batches, device,
                                    n_embd, bits=4, lr=1e-4, steps_per_block=8,
                                    group_size=128, damp_percent=0.01,
                                    grad_clip_norm=1.0, verbose=True):
    """
    Objective 1 (`min_{W_hat_Q,W_hat_K,W_hat_V} L(A(X), A_hat(X))`), made robust:

      1. GPTQ-warm-starts each block INSIDE this function (instead of a
         separate pre-quantization pass) so we can capture the EXACT
         per-(output-channel, group) scale GPTQ used via `return_scale=True`.
         This is the fix for the bug described in the markdown above: the
         old code re-derived a scale with a different granularity/axis than
         GPTQ's own scale, silently corrupting the warm start before any
         fine-tuning step ran.
      2. Tracks the BEST iterate seen during each block's fine-tuning (by
         attention_loss on that block's calibration batches), not just the
         last one -- an ~8-step Adam trajectory is noisy.
      3. Safety net: a block's fine-tuned weights are committed only if they
         beat the GPTQ-only starting loss; otherwise the block keeps its
         GPTQ-only weights. Formally guarantees
         attention_loss(final) <= attention_loss(GPTQ-only warm start)
         for every block (a guarantee on the local reconstruction objective,
         which is itself a proxy for perplexity -- not a hard guarantee on
         perplexity -- but it removes this specific failure mode).
      4. Gradient clipping, and calibration batches that rotate across the
         calibration set per block instead of every block reusing the same
         first `steps_per_block` batches.

    `target_A` always comes from `model_float` (untouched, full precision).
    `X` (this block's input activations) comes from `model_joint`, so later
    blocks train against realistic post-quantization-error inputs from
    earlier blocks, consistent with how GPTQ itself behaves sequentially.
    """
    n_blocks = len(model_joint.transformer.h)

    for block_idx in range(n_blocks):
        block = model_joint.transformer.h[block_idx]
        c_attn = block.attn.c_attn

        # --- GPTQ warm start, done HERE so we can capture its exact scale ---
        H = collect_hessian_via_hook(model_joint, c_attn, calibration_batches, device)
        _, scale_full = gptq_quantize_layer(
            c_attn.weight.data, H, bits=bits, group_size=group_size,
            damp_percent=damp_percent, act_order=True, return_scale=True,
        )  # c_attn.weight.data is now GPTQ-quantized in place

        W = c_attn.weight.data
        W_Q0, W_K0, W_V0 = W.split(n_embd, dim=1)
        w_flat = torch.cat([W_Q0.flatten(), W_K0.flatten(), W_V0.flatten()]).clone()
        w_flat.requires_grad_(True)

        # the EXACT scale GPTQ used -- ste_quantize(w_flat, scale_flat, bits)
        # reproduces w_flat exactly right now, before any Adam step.
        scale_Q, scale_K, scale_V = scale_full.split(n_embd, dim=1)
        scale_flat = torch.cat([scale_Q.flatten(), scale_K.flatten(), scale_V.flatten()])

        bias = c_attn.bias.data
        b_Q, b_K, b_V = bias.split(n_embd)

        optimizer = torch.optim.Adam([w_flat], lr=lr)

        # rotate through the calibration set instead of every block reusing
        # the same first `steps_per_block` batches
        start = (block_idx * steps_per_block) % len(calibration_batches)
        idxs = [(start + s) % len(calibration_batches) for s in range(steps_per_block)]
        batches = [calibration_batches[i] for i in idxs]

        with torch.no_grad():
            init_losses = []
            for batch in batches:
                _, target_A, _ = get_calibration_data_for_block(model_float, batch, device, block_idx)
                X, _, _ = get_calibration_data_for_block(model_joint, batch, device, block_idx)
                init_losses.append(attention_loss(w_flat.detach(), X, target_A, n_embd,
                                                    b_Q=b_Q, b_K=b_K, b_V=b_V).item())
            init_loss = sum(init_losses) / len(init_losses)

        best_loss = init_loss
        best_w = w_flat.detach().clone()

        for batch in batches:
            _, target_A, _ = get_calibration_data_for_block(model_float, batch, device, block_idx)
            X, _, _ = get_calibration_data_for_block(model_joint, batch, device, block_idx)

            optimizer.zero_grad()
            w_q = ste_quantize(w_flat, scale_flat, bits)
            loss = attention_loss(w_q, X, target_A, n_embd, b_Q=b_Q, b_K=b_K, b_V=b_V)
            loss.backward()
            torch.nn.utils.clip_grad_norm_([w_flat], grad_clip_norm)
            optimizer.step()

            loss_val = loss.item()
            if loss_val < best_loss:
                best_loss = loss_val
                with torch.no_grad():
                    best_w = ste_quantize(w_flat, scale_flat, bits).detach().clone()

        if best_loss <= init_loss:
            W_Q, W_K, W_V = reshape_weights(best_w, n_embd)
            c_attn.weight.data.copy_(torch.cat([W_Q, W_K, W_V], dim=1))
            status = f"improved {init_loss:.6f} -> {best_loss:.6f}"
        else:
            # best_w never beat the GPTQ-only starting point; c_attn.weight.data
            # is already that GPTQ-only solution (nothing else has touched it),
            # so we simply don't overwrite it with the worse fine-tuned weights.
            status = f"kept GPTQ-only ({init_loss:.6f}; fine-tune best was {best_loss:.6f})"

        if verbose:
            print(f"  Block {block_idx}: {status}")

    return model_joint


In [ ]:
print("Loading a fresh full-precision GPT-2 as the fixed reference (target_A source)...")
model_float = AutoModelForCausalLM.from_pretrained("gpt2").to(DEVICE)
model_float.eval()

print("Loading a second, still full-precision GPT-2 for joint attention-aware fine-tuning...")
print("(GPTQ warm-start now happens INSIDE joint_attention_aware_finetune, per block,")
print(" so the exact scale it used can be reused for fine-tuning -- see the fix above.)")
model_joint = AutoModelForCausalLM.from_pretrained("gpt2").to(DEVICE)
model_joint.eval()

print("\nRunning joint attention-aware fine-tuning (Objective 1: minimizes L(A(X), A_hat(X)) directly)...")
model_joint = joint_attention_aware_finetune(model_joint, model_float, calibration_batches, DEVICE,
                                              n_embd, bits=4, lr=1e-4, steps_per_block=8,
                                              group_size=128)

print("\nEvaluating perplexity...")
ppl_joint = evaluate_perplexity(model_joint, tokenizer)
print(f"\nJoint attention-aware fine-tuned (Objective 1) perplexity: {ppl_joint:.3f}")


Loading a fresh full-precision GPT-2 as the fixed reference (target_A source)...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading a second, still full-precision GPT-2 for joint attention-aware fine-tuning...
(GPTQ warm-start now happens INSIDE joint_attention_aware_finetune, per block,
 so the exact scale it used can be reused for fine-tuning -- see the fix above.)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Running joint attention-aware fine-tuning (Objective 1: minimizes L(A(X), A_hat(X)) directly)...
  Block 0: improved 0.000412 -> 0.000392
  Block 1: improved 0.001787 -> 0.001695
  Block 2: improved 0.003554 -> 0.003453
  Block 3: improved 0.005943 -> 0.005595
  Block 4: improved 0.005079 -> 0.004761
  Block 5: improved 0.009606 -> 0.008352
  Block 6: improved 0.012166 -> 0.010587
  Block 7: improved 0.012789 -> 0.011700
  Block 8: improved 0.015866 -> 0.013312
  Block 9: improved 0.017130 -> 0.015142
  Block 10: improved 0.015347 -> 0.013961
  Block 11: improved 0.024203 -> 0.021330

Evaluating perplexity...

Joint attention-aware fine-tuned (Objective 1) perplexity: 26.225


## 12. Compare results

In [ ]:
print(f"Uniform  4-bit GPTQ perplexity              : {ppl_uniform:.3f}")
print(f"Adaptive JAB-Hessian perplexity              : {ppl_adaptive:.3f}")
print(f"Joint attention-aware fine-tuned perplexity  : {ppl_joint:.3f}")
print(f"Adaptive vs. uniform difference               : {ppl_adaptive - ppl_uniform:+.3f}")
print(f"Joint    vs. uniform difference               : {ppl_joint - ppl_uniform:+.3f}")


Uniform  4-bit GPTQ perplexity              : 26.225
Adaptive JAB-Hessian perplexity              : 26.225
Joint attention-aware fine-tuned perplexity  : 26.225
Adaptive vs. uniform difference               : +0.000
Joint    vs. uniform difference               : +0.000
